In [10]:
import duckdb
import logging
import sys

In [11]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
)
logger = logging.getLogger(__name__)

In [14]:
def load_secop_data(csv_path: str, db_path: str = "secop.duckdb"):

    # con = duckdb.connect(db_path)
    with duckdb.connect(db_path) as con:

        logger.info(f"--- Iniciando procesamiento de {csv_path} ---")

        # 1. Configurar límites para proteger la RAM
        con.execute("SET memory_limit = '2GB';")
        con.execute("SET temp_directory = '.duckdb_temp';") # Directorio para desbordamiento a disco

        # 2. Crear una VISTA sobre el CSV (no carga datos en memoria aún)
        # read_csv_auto detecta tipos y procesa por chunks automáticamente
        con.execute(f"""
        CREATE OR REPLACE VIEW stg_raw_secop AS
        SELECT * FROM read_csv_auto('{csv_path}', all_varchar=True);
        """)

        logger.info("Vista creada. Procesando Entidades...")
        # 3. Crear tablas intermedias deduplicadas
        con.execute("""
        CREATE TABLE IF NOT EXISTS entidades AS
        SELECT DISTINCT
        "Nit Entidad" AS nit,
        "Entidad" AS nombre,
        "Departamento Entidad" AS departamento,
        "Ciudad Entidad" AS ciudad,
        "Codigo Entidad" AS codigo_entidad
        FROM stg_raw_secop
        WHERE "Nit Entidad" IS NOT NULL;
        """)

        logger.info("Procesando Proveedores...")
        con.execute("""
            CREATE TABLE IF NOT EXISTS proveedores AS
            SELECT DISTINCT
                "NIT del Proveedor Adjudicado" AS nit,
                "Nombre del Proveedor Adjudicado" AS nombre,
                "CodigoProveedor" AS codigo_proveedor
            FROM stg_raw_secop
            WHERE "NIT del Proveedor Adjudicado" IS NOT NULL
              AND "NIT del Proveedor Adjudicado" != 'No Definido';
        """)

        logger.info("Procesando Procesos...")
        con.execute("""
            CREATE TABLE IF NOT EXISTS procesos AS
            SELECT
                "ID del Proceso" AS id_proceso,
                "Nit Entidad" AS nit_entidad,
                "NIT del Proveedor Adjudicado" AS nit_proveedor,
                "Precio Base" AS precio_base,
                "Estado del Procedimiento" AS estado,
                "Fecha de Publicacion del Proceso" AS fecha_publicacion,
                "URLProceso" AS url
            FROM stg_raw_secop;
        """)

        # Verificar resultados
        count = con.execute("SELECT count(*) FROM procesos").fetchone()[0]
        logger.info(f"--- Finalizado ---")
        logger.info(f"Total de procesos cargados: {count}")

In [8]:
if __name__ == "__main__":
    load_secop_data("secip_ii.csv")

--- Iniciando procesamiento de secip_ii.csv ---


IOException: IO Error: No files found that match the pattern "secip_ii.csv"

LINE 3:         SELECT * FROM read_csv_auto('secip_ii.csv', all_varchar=True);
                              ^